# 02 — Build the project datasets

This notebook turns the raw Food.com files into the smaller, cleaned datasets the team works with:

| File | What it is | Used for |
|---|---|---|
| `data/processed/recipes.csv` | every recipe in one of the 9 shortlisted cuisines, with **one** simple `cuisine` label | networks |
| `data/processed/reviews.csv` | the cleaned reviews of exactly those recipes | NLP |
| `data/processed/recipes_mini.csv` | a random 30% of the recipes of **each** cuisine (70% removed) | networks, when the full set is too slow |
| `data/processed/reviews_mini.csv` | the reviews of the mini recipes | NLP on the mini set |

The cuisine shortlist and the problems cleaned here were found in `01_initial_eda.ipynb`.

**Only want to use the datasets?** Skip to Section 8, which shows how to load them and how to keep only some cuisines.

## 0. Setup

In [1]:
import ast
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

SHORTLIST = ["italian", "mexican", "indian", "greek", "french", "chinese", "middle-eastern", "thai", "japanese"]
MINI_KEEP_FRACTION = 0.30   # keep 30% of each cuisine = randomly remove 70%
RNG_SEED = 42
pd.set_option("display.max_colwidth", 90)

steps_log = []  # one row per cleaning step, shown in Section 5
def log_step(table, step, change, rows):
    steps_log.append({"table": table, "step": step, "change": change, "rows": rows})

## 1. Load the raw files
The columns `tags`, `ingredients` and `steps` are stored as text that looks like a Python list, so we parse them with `ast.literal_eval` (as in notebook 01). Each review gets a `review_id`, its row number in `RAW_interactions.csv`, so any review can be traced back to the raw file.

In [2]:
recipes_raw = pd.read_csv(RAW / "RAW_recipes.csv", parse_dates=["submitted"])
reviews_raw = pd.read_csv(RAW / "RAW_interactions.csv", parse_dates=["date"])
reviews_raw = reviews_raw.rename_axis("review_id").reset_index()

for col in ["tags", "ingredients", "steps"]:
    recipes_raw[col] = recipes_raw[col].map(ast.literal_eval)

print(f"raw recipes: {len(recipes_raw):,} | raw reviews: {len(reviews_raw):,}")

raw recipes: 231,637 | raw reviews: 1,132,367


## 2. One simple cuisine label per recipe
A recipe belongs to a cuisine if it has that tag. Food.com also has sub-cuisine tags such as `szechuan`, `baja` or `lebanese`, which notebook 01 found *under* `chinese`, `mexican` and `middle-eastern`. The check below confirms that every recipe with a sub-tag also has the main tag. So we only look at the 9 main tags, and the label is always the simple one: a `szechuan` recipe is just `chinese`.

In [3]:
SUBTAGS = {
    "chinese": ["szechuan", "cantonese", "hunan", "beijing", "mongolian"],
    "mexican": ["baja", "oaxacan"],
    "middle-eastern": ["lebanese", "turkish", "iranian-persian", "jewish-sephardi", "saudi-arabian",
                       "iraqi", "palestinian", "georgian"],
}
check = pd.DataFrame([
    {"main tag": main, "sub-tag": sub,
     "recipes with sub-tag": recipes_raw["tags"].map(lambda t, s=sub: s in t).sum(),
     "of which without the main tag": recipes_raw["tags"].map(lambda t, s=sub, m=main: s in t and m not in t).sum()}
    for main, subs in SUBTAGS.items() for sub in subs
])
assert check["of which without the main tag"].sum() == 0
check

,main tag,sub-tag,recipes with sub-tag,of which without the main tag
0,chinese,szechuan,114,0
1,chinese,cantonese,90,0
2,chinese,hunan,31,0
3,chinese,beijing,18,0
4,chinese,mongolian,13,0
5,mexican,baja,160,0
6,mexican,oaxacan,72,0
7,middle-eastern,lebanese,308,0
8,middle-eastern,turkish,286,0
9,middle-eastern,iranian-persian,252,0


Recipes with **two** of the 9 cuisines (for example `italian` and `french`) are dropped, so that every recipe counts for exactly one cuisine.

In [4]:
in_cuisine = pd.DataFrame({c: recipes_raw["tags"].map(lambda t, c=c: c in t) for c in SHORTLIST})
n_cuisines = in_cuisine.sum(axis=1)

recipes = recipes_raw[n_cuisines == 1].copy()
recipes["cuisine"] = in_cuisine[n_cuisines == 1].idxmax(axis=1)

log_step("recipes", "recipes in one of the 9 cuisines", "start", int((n_cuisines >= 1).sum()))
log_step("recipes", "drop recipes in two or more of the 9 cuisines", f"−{(n_cuisines >= 2).sum():,} rows", len(recipes))
recipes["cuisine"].value_counts().to_frame("recipes")

,recipes
cuisine,
italian,7218
mexican,6657
indian,2658
french,2186
greek,2182
middle-eastern,1939
chinese,1927
thai,1172
japanese,790


## 3. Clean the recipes
A recipe is only **removed** if its ingredient list is unusable, because the networks are built from ingredients. An implausible cooking time is set to missing (`NaN`) instead, so we don't lose a good recipe because of one bad number.

### 3.1 Recipes with fewer than 2 ingredients
A recipe with a single ingredient has no ingredient pairs, so it adds nothing to the co-occurrence network.

In [5]:
few = recipes["ingredients"].map(len) < 2
display(recipes.loc[few, ["id", "name", "cuisine", "ingredients"]])
recipes = recipes[~few]
log_step("recipes", "drop recipes with fewer than 2 ingredients", f"−{few.sum():,} rows", len(recipes))

,id,name,cuisine,ingredients
106735,376622,homemade panko japanese bread crumbs,japanese,[white bread]


### 3.2 Duplicate recipes
Two recipes are duplicates if they have the same name **and** the same set of ingredients. We keep the one posted first.

In [6]:
recipes = recipes.sort_values(["submitted", "id"])  # so that keep="first" keeps the earliest post
key = recipes["name"].fillna("") + "||" + recipes["ingredients"].map(lambda l: "|".join(sorted(set(l))))
dupe = key.duplicated(keep="first")
display(recipes.loc[key.duplicated(keep=False), ["id", "name", "cuisine", "submitted", "n_ingredients"]])
recipes = recipes[~dupe].sort_index()
log_step("recipes", "drop duplicate recipes (same name and ingredients)", f"−{dupe.sum():,} rows", len(recipes))

,id,name,cuisine,submitted,n_ingredients
111208,292756,indian style green beans,indian,2008-03-19,9
111141,426090,indian style green beans,indian,2010-05-17,9


### 3.3 Cooking time (`minutes`)
* `minutes == 0` is not a real cooking time; the author did not fill it in → `NaN`.
* For very long times we use Tukey's outlier rule. Cooking times are very skewed, so we apply it to the log of the time. A time is an outlier if log10(minutes) > Q3 + 3 × (Q3 − Q1), where Q1 and Q3 are the 25th and 75th percentiles. Short times (1–2 minutes for a drink) are normal, so there is no lower limit.

Some flagged recipes really are slow (liqueurs, preserves), which is why we only blank the time and keep the recipe.

In [7]:
log_min = np.log10(recipes.loc[recipes["minutes"] > 0, "minutes"])
q1, q3 = log_min.quantile([0.25, 0.75])
minutes_limit = 10 ** (q3 + 3 * (q3 - q1))

zero_min = recipes["minutes"] == 0
long_min = recipes["minutes"] > minutes_limit
print(f"Q1 = {10**q1:.0f} min, Q3 = {10**q3:.0f} min → upper limit = {minutes_limit:,.0f} min (≈ {minutes_limit / 60:.0f} hours)")
display(recipes.loc[long_min, ["name", "cuisine", "minutes"]].sort_values("minutes", ascending=False).head(5))

recipes["minutes"] = recipes["minutes"].astype(float).mask(zero_min | long_min)
log_step("recipes", "minutes == 0 → NaN", f"{zero_min.sum():,} values blanked", len(recipes))
log_step("recipes", f"minutes > {minutes_limit:,.0f} → NaN", f"{long_min.sum():,} values blanked", len(recipes))

Q1 = 25 min, Q3 = 65 min → upper limit = 1,142 min (≈ 19 hours)


,name,cuisine,minutes
170999,raspberry liqueur,french,172810
145524,nut wine,french,132495
124353,limoncello lemoncella,italian,115230
124351,limoncello italian lemoncello or orangecello,italian,86400
84832,flavored vinegar,french,64815


## 4. Clean the reviews
We start from the reviews of the recipes kept above.

In [8]:
reviews = reviews_raw.merge(recipes[["id", "cuisine", "contributor_id"]],
                            left_on="recipe_id", right_on="id", how="inner").drop(columns="id")
log_step("reviews", "reviews of the kept recipes", "start", len(reviews))
print(f"reviews of the kept recipes: {len(reviews):,}")

reviews of the kept recipes: 145,850


### 4.1 Reviews written for community games
Many reviews were written to score points in site games: Zaar World Tour, Pick A Chef, Recipe Swap, many *Tag* games, and others. They are not ordinary feedback, and Zaar World Tour sends players to foreign cuisines on purpose (notebook 01), so we remove them. The game names come from reading the most common "Made for ..." phrases. Two words need care because they also occur in normal reviews: *swap* ("swap the sugar for honey") and *tag* ("price tag", "tag along"), so the patterns exclude those uses.

In [9]:
GAME_PATTERNS = {
    "Zaar World Tour":   r"\bzwt ?\d*\b|\bwzt ?\d*\b|zaar world tour|\bworld tour ?\d+|\bwt ?\d\b",
    "Pick A Chef":       r"\bpac\b|pick[ -]a[ -]chef",
    "Recipe Swap":       r"\bswap\b(?!\s+(?:out|around|in|for|it|the|them|with|that|this|some|a|an|to)\b)",
    "Tag games":         r"(?<!price )\btag(?:ged)?\b(?! along)|\bprmrt?\b",
    "Culinary Quest":    r"culinary quest|\bcq ?\d+\b",
    "1-2-3 Hit Wonders": r"hit wonders|\b1-?2-?3 hits\b",
    "Zaar Stars":        r"zaar stars?\b",
    "My 3 Chefs":        r"\bmy (?:3|three) chefs\b",
    "other games":       r"please review my recipe|everyday is a holiday|comfort cafe|all you can cook buffet|"
                         r"healthy choices abc|bargain basement|new kids on the block|zaar chef alphabet soup",
}
GAME_ANY = re.compile("|".join(GAME_PATTERNS.values()))

text = reviews["review"].fillna("").str.lower()
game_hits = pd.DataFrame({name: text.str.contains(p, regex=True) for name, p in GAME_PATTERNS.items()})
is_game = game_hits.any(axis=1)

print(f"game reviews: {is_game.sum():,} ({is_game.mean():.1%})")
pd.DataFrame({
    "reviews matched": game_hits.sum(),
}).T

game reviews: 12,981 (8.9%)


,Zaar World Tour,Pick A Chef,Recipe Swap,Tag games,Culinary Quest,1-2-3 Hit Wonders,Zaar Stars,My 3 Chefs,other games
reviews matched,5406,1847,1046,3772,364,189,141,139,737


In [10]:
# Spot check: 10 random game reviews, cut around the matching words
for t in reviews.loc[is_game, "review"].sample(10, random_state=RNG_SEED):
    m = GAME_ANY.search(t.lower())
    start = max(0, m.start() - 70)
    print("…" + t[start:m.end() + 50].replace("\n", " ") + "…")

… first drawn to this recipe by Dreamer in Ontario's photograph in the PRMR thread and WOW!  It was just so totally delicious…
…d fresh mozzarella.  Thanks MizEmerilLagasse :)  Made for Newest Zaar tag game…
…ad!  It was very refreshing and colorful.  Thanks for sharing!  Veg*n Swap 25…
…ed ingredient, in addition to, the parmesan cheese.  Made for the FYC tag game.…
… liked the zip of the mint.  Thanks for sharing the recipe!  Made for ZWT9, The Apron String Travelers…
…e who liked it more.  Thanks Littlemafia :)  Made for Sweet tradition tag game…
…I made this last night for ZWT2, Canada. YUM!! Ohhh... I ate myself into a comfor…
…Made for Zaar Alphabet Soup Tag game.  This was very very good.  I cut the recipe…
…lly be at least 4 stars as written.  Thank you for posting.  Made for ZWT5 for the Groovy GastroGnomes!…
 Made for Spring PAC 08'…


In [11]:
print("share of game reviews per cuisine:")
display((is_game.groupby(reviews["cuisine"]).mean() * 100).round(1).sort_values(ascending=False).to_frame("% game reviews").T)

reviews = reviews[~is_game]
log_step("reviews", "drop game reviews", f"−{is_game.sum():,} rows", len(reviews))

share of game reviews per cuisine:


cuisine,greek,french,middle-eastern,indian,thai,mexican,italian,japanese,chinese
% game reviews,15.3,14.0,13.0,10.3,9.0,7.8,6.7,6.7,6.3


### 4.2 Author replies, empty reviews and rating 0
* A "review" by the recipe's own author is a reply to other users, not a review → removed.
* Reviews without text are useless for NLP → removed.
* A rating of 0 means "no stars given", not "worst recipe" (notebook 01) → the rating is set to missing, the text is kept.

In [12]:
own = reviews["user_id"] == reviews["contributor_id"]
reviews = reviews[~own]
log_step("reviews", "drop replies by the recipe's author", f"−{own.sum():,} rows", len(reviews))

empty = reviews["review"].fillna("").str.strip() == ""
reviews = reviews[~empty]
log_step("reviews", "drop reviews without text", f"−{empty.sum():,} rows", len(reviews))

zero = reviews["rating"] == 0
reviews["rating"] = reviews["rating"].astype("Int64").mask(zero)
log_step("reviews", "rating 0 → missing", f"{zero.sum():,} values blanked", len(reviews))
print(f"reviews left: {len(reviews):,}")

reviews left: 132,744


## 5. Cleaning summary
Each row is one step. `change` says whether the step removed rows or only blanked values; `rows` is the number of rows left after it.

In [13]:
pd.DataFrame(steps_log)

,table,step,change,rows
0,recipes,recipes in one of the 9 cuisines,start,27160
1,recipes,drop recipes in two or more of the 9 cuisines,−431 rows,26729
2,recipes,drop recipes with fewer than 2 ingredients,−1 rows,26728
3,recipes,drop duplicate recipes (same name and ingredients),−1 rows,26727
4,recipes,minutes == 0 → NaN,128 values blanked,26727
5,recipes,"minutes > 1,142 → NaN",258 values blanked,26727
6,reviews,reviews of the kept recipes,start,145850
7,reviews,drop game reviews,"−12,981 rows",132869
8,reviews,drop replies by the recipe's author,−107 rows,132762
9,reviews,drop reviews without text,−18 rows,132744


## 6. Final and mini datasets
### 6.1 Columns
We keep only what the network and NLP parts need. The raw `tags` column is dropped because it contains the sub-cuisine tags, so `cuisine` is the only cuisine column. Recipes whose reviews were all removed stay in the recipe file with `n_reviews = 0`; the networks don't need reviews.

In [14]:
recipes["n_reviews"] = recipes["id"].map(reviews["recipe_id"].value_counts()).fillna(0).astype(int)

RECIPE_COLS = ["recipe_id", "name", "cuisine", "ingredients", "n_ingredients", "steps", "n_steps",
               "description", "minutes", "submitted", "contributor_id", "n_reviews"]
REVIEW_COLS = ["review_id", "recipe_id", "cuisine", "user_id", "date", "rating", "review"]

recipes_final = (recipes.rename(columns={"id": "recipe_id"})[RECIPE_COLS]
                 .sort_values(["cuisine", "recipe_id"]).reset_index(drop=True))
reviews_final = reviews[REVIEW_COLS].sort_values(["cuisine", "recipe_id", "date"]).reset_index(drop=True)
print(f"recipes: {len(recipes_final):,} | reviews: {len(reviews_final):,}")
recipes_final.head(3)

recipes: 26,727 | reviews: 132,744


,recipe_id,name,cuisine,ingredients,n_ingredients,steps,n_steps,description,minutes,submitted,contributor_id,n_reviews
0,195,chinese hot and sour chicken noodle salad,chinese,"[chicken breast, dry linguine, peppercorn, dried hot red chile, salad oil, rice vinega...",9,"[in a 5-6 quart pan , bring about 3 quarts of water to a boil, add chicken , cover and...",22,originally from,80.0,1999-08-27,1545,1
1,198,chinese hot and sour pork soup,chinese,"[chicken stock, lean pork, garlic and red chile paste, soy sauce, white pepper, egg su...",17,"[bring stock to a simmer, add soy sauce , pork , mushrooms and chile paste, simmer for...",11,edited recipe from other viewers recomendations,40.0,1999-09-18,1545,49
2,203,chinese plum sauce,chinese,"[plums, onion, water, gingerroot, garlic, granulated sugar, rice vinegar, coriander, s...",12,"[in large heavy saucepan , bring plums , onions , water , ginger and garlic to boil ov...",9,chinese plum sauce serve this with egg rolls or brush it over chicken and ribs on the...,115.0,1999-08-06,1533,16


In [15]:
reviews_final.head(3)

,review_id,recipe_id,cuisine,user_id,date,rating,review
0,623253,195,chinese,868654,2010-06-12,5,Fairly simple to make and the outcome is a delicious salad that makes you want another...
1,671682,198,chinese,5067,2001-01-16,5,Great recipe for anyone who like it as close as it come to the real thing! Very simple...
2,671683,198,chinese,5227,2001-01-23,4,my family like the bamboo shoots whole. but we love your recipe


### 6.2 Mini dataset
For each cuisine separately we keep a random 30% of its recipes (70% removed). Sampling inside each cuisine keeps the cuisine proportions of the full dataset, and the fixed seed gives everyone the same mini dataset. The mini reviews are the reviews of the mini recipes.

In [16]:
recipes_mini = (recipes_final.groupby("cuisine", group_keys=False)
                .sample(frac=MINI_KEEP_FRACTION, random_state=RNG_SEED)
                .sort_values(["cuisine", "recipe_id"]).reset_index(drop=True))
reviews_mini = reviews_final[reviews_final["recipe_id"].isin(recipes_mini["recipe_id"])].reset_index(drop=True)

def per_cuisine(rec, rev):
    t = pd.DataFrame({
        "recipes": rec["cuisine"].value_counts(),
        "reviews": rev["cuisine"].value_counts(),
        "distinct ingredients": rec.explode("ingredients").groupby("cuisine")["ingredients"].nunique(),
    })
    t.loc["TOTAL"] = [len(rec), len(rev), rec["ingredients"].explode().nunique()]
    return t

comparison = pd.concat({"full": per_cuisine(recipes_final, reviews_final),
                        "mini": per_cuisine(recipes_mini, reviews_mini)}, axis=1)
comparison[("mini", "% of full recipes")] = (comparison[("mini", "recipes")] / comparison[("full", "recipes")] * 100).round(1)
comparison

full                                 mini          \
               recipes reviews distinct ingredients recipes reviews   
cuisine                                                               
chinese           1927   10966                 1810     578    2766   
french            2186    8347                 2383     656    2686   
greek             2182   10830                 1907     655    2429   
indian            2657    9334                 2024     797    2661   
italian           7218   40272                 3657    2165   13291   
japanese           789    4141                 1125     237    1007   
mexican           6657   36868                 3456    1997    9820   
middle-eastern    1939    7064                 1826     582    2122   
thai              1172    4922                 1428     352    1396   
TOTAL            26727  132744                 7296    8019   38178   

                                                       
               distinct ingredients % of full recipes  
cuisine                                                
chinese                        1041              30.0  
french                         1311              30.0  
greek                          1046              30.0  
indian                         1189              30.0  
italian                        2215              30.0  
japanese                        599              30.0  
mexican                        2039              30.0  
middle-eastern                 1017              30.0  
thai                            846              30.0  
TOTAL                          4827              30.0

## 7. Checks and export
These checks stop the notebook if any promise about the datasets is broken.

In [17]:
for name, rec, rev in [("full", recipes_final, reviews_final), ("mini", recipes_mini, reviews_mini)]:
    assert rec["recipe_id"].is_unique and rev["review_id"].is_unique, name
    assert set(rec["cuisine"]) == set(SHORTLIST), name                         # the 9 cuisines, nothing else
    assert rev["recipe_id"].isin(rec["recipe_id"]).all(), name                 # every review has its recipe
    assert (rev["cuisine"] == rev["recipe_id"].map(rec.set_index("recipe_id")["cuisine"])).all(), name
    assert (rec["ingredients"].map(len) >= 2).all(), name
    assert (rev["review"].str.strip() != "").all(), name
    assert not (rev["rating"] == 0).any(), name
    assert not rev["review"].str.lower().str.contains(GAME_ANY).any(), name    # no game reviews left
assert recipes_mini["recipe_id"].isin(recipes_final["recipe_id"]).all()        # mini is part of full
share = recipes_mini["cuisine"].value_counts() / recipes_final["cuisine"].value_counts()
assert share.between(0.29, 0.31).all(), share                                 # ~30% of every cuisine
print("all checks passed")

all checks passed


The files are CSV so they open anywhere. The list columns `ingredients` and `steps` are written as JSON lists. These four files are committed to the repository (the raw files are not).

In [18]:
def export(df, name):
    df = df.copy()
    for col in ["ingredients", "steps"]:
        if col in df:
            df[col] = df[col].map(json.dumps)
    path = OUT / f"{name}.csv"
    df.to_csv(path, index=False)
    return {"file": f"data/processed/{name}.csv", "rows": len(df), "MB": round(path.stat().st_size / 1e6, 1)}

pd.DataFrame([export(recipes_final, "recipes"), export(reviews_final, "reviews"),
              export(recipes_mini, "recipes_mini"), export(reviews_mini, "reviews_mini")])

,file,rows,MB
0,data/processed/recipes.csv,26727,31.3
1,data/processed/reviews.csv,132744,43.0
2,data/processed/recipes_mini.csv,8019,9.3
3,data/processed/reviews_mini.csv,38178,12.4


## 8. How to use the datasets
### 8.1 Load them
Copy this function into your notebook. It turns the JSON lists back into Python lists and the dates back into dates. Use `"recipes_mini"` / `"reviews_mini"` for the mini dataset.

In [19]:
DATA = ROOT / "data" / "processed"

def load(name):
    df = pd.read_csv(DATA / f"{name}.csv")
    for col in ["submitted", "date"]:
        if col in df:
            df[col] = pd.to_datetime(df[col])
    for col in ["ingredients", "steps"]:
        if col in df:
            df[col] = df[col].map(json.loads)
    if "rating" in df:
        df["rating"] = df["rating"].astype("Int64")
    return df

recipes = load("recipes")
reviews = load("reviews")

# Check: what we read back is exactly what we exported
pd.testing.assert_frame_equal(recipes, recipes_final, check_dtype=False)
pd.testing.assert_frame_equal(reviews, reviews_final, check_dtype=False)
print(f"recipes: {len(recipes):,} | reviews: {len(reviews):,}")
recipes[["recipe_id", "name", "cuisine", "ingredients"]].head(3)

recipes: 26,727 | reviews: 132,744


,recipe_id,name,cuisine,ingredients
0,195,chinese hot and sour chicken noodle salad,chinese,"[chicken breast, dry linguine, peppercorn, dried hot red chile, salad oil, rice vinega..."
1,198,chinese hot and sour pork soup,chinese,"[chicken stock, lean pork, garlic and red chile paste, soy sauce, white pepper, egg su..."
2,203,chinese plum sauce,chinese,"[plums, onion, water, gingerroot, garlic, granulated sugar, rice vinegar, coriander, s..."


### 8.2 Keep only some cuisines
Example: a smaller dataset with only Mexican, Japanese and Italian recipes. Filter the recipes on `cuisine`, then keep the reviews of those recipes. Because every review also has a `cuisine` column, filtering the reviews on `cuisine` gives the same result.

In [20]:
def keep_cuisines(recipes, reviews, cuisines):
    rec = recipes[recipes["cuisine"].isin(cuisines)].reset_index(drop=True)
    rev = reviews[reviews["recipe_id"].isin(rec["recipe_id"])].reset_index(drop=True)
    return rec, rev

my_recipes, my_reviews = keep_cuisines(recipes, reviews, ["mexican", "japanese", "italian"])

print(f"recipes: {len(my_recipes):,} | reviews: {len(my_reviews):,}")
pd.DataFrame({"recipes": my_recipes["cuisine"].value_counts(),
              "reviews": my_reviews["cuisine"].value_counts()})

recipes: 14,664 | reviews: 81,281


,recipes,reviews
cuisine,,
italian,7218,40272
mexican,6657,36868
japanese,789,4141


The same works on the mini dataset: `keep_cuisines(load("recipes_mini"), load("reviews_mini"), [...])`.